# 📓 Semana 12 · Dia 3 — Busca híbrida (BM25 + semântica) com RRF e reranking

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (conceito) + 🔑 rerank (trial) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Recall@5 melhorado com híbrido + rerank |

---


## 📖 Teoria — Por que híbrido?

A busca semântica erra nomes próprios/códigos (ex.: '85123A'); a busca lexical (BM25) erra sinônimos. A **busca híbrida** combina as duas e funde com **RRF (Reciprocal Rank Fusion)**:

```
score RRF = Σ 1/(k + posição_do_item)
```

O **reranking** reordena os top-N com um cross-encoder (modelo que compara pergunta×chunk de uma vez) — qualidade bem melhor, custo de latência.


### 💻 Na prática — Busca híbrida

Combine resultados do Vector Search com BM25.


In [ ]:
# BM25 (lexical) — usando a própria tabela como fonte de texto
from pyspark.sql.functions import col, lower
from pyspark.ml.feature import Tokenizer
df = spark.table("workspace.prata.produtos_rag")
df_lex = df.withColumn("texto_lower", lower(col("texto")))
print("Base lexical pronta (busca por palavras exatas).")

In [ ]:
# Busca semântica (vector)
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient()
sem = vsc.similarity_search(index_name="workspace.prata.produtos_rag_index",
                            query_text="copo de vidro",
                            columns=["StockCode"], num_results=10)
print("Semântica:", [r[0] for r in sem["result"]["data_array"]])

In [ ]:
# Fusão RRF (implementação didática)
def rrf(*listas, k=60):
    scores = {}
    for lst in listas:
        for pos, item in enumerate(lst, start=1):
            scores[item] = scores.get(item, 0) + 1 / (k + pos)
    return sorted(scores, key=scores.get, reverse=True)
sem_lista = [r[0] for r in sem["result"]["data_array"]]
lex_lista = ["85123A", "71053", "22423"]  # resultados BM25 (exemplo)
print("Fusão RRF:", rrf(sem_lista, lex_lista)[:5])

### 💻 Na prática — Reranking

Na Free Edition, o rerank via cross-encoder é conceitual (🔑 no trial). O padrão: endpoint de rerank do Databricks recebe pergunta + chunks e retorna scores.


In [ ]:
# Rerank (conceito; 🔑 no trial)
print("""
1. Endpoint de rerank (Mosaic AI Reranker, 🔑)
2. Entrada: query + lista de chunks (top-10)
3. Saída: mesma lista reordenada por relevância
4. Use os top-3 reordenados como contexto final
""")
print("Rerank melhora Recall@5 em ~10-25% — teste no trial.")

> 🎯 **Dica de prova**: GenAI Assoc: RRF (fórmula 1/(k+rank)), híbrido = lexical + semântico, rerank = cross-encoder no top-N. Pergunta: 'como combinar BM25 e vetores?' → RRF.


## 🎯 Exercícios de fixação

**1.** Explique a fórmula do RRF com um exemplo de 2 listas.

**2.** Quando o BM25 vence a busca semântica?

**3.** Por que rerank só no top-N e não em tudo?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** RRF

Se um item está em posição 1 nas duas listas: 1/61 + 1/61 = 0.033 — itens bem rankeados nas duas vencem.

**2.** BM25 vence

Códigos, IDs, nomes exatos e siglas — onde o 'som' importa mais que o significado.

**3.** Top-N

Cross-encoder é caro (compara par a par); aplicá-lo em 10–20 candidatos dá o ganho com latência aceitável.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*